### 347. Top K Frequent Elements
Given an integer array `nums` and an integer `k`, return the `k` most frequent elements. You may return the answer in any order.

**Example 1**:
```shell
Input: nums = [1,1,1,2,2,3], k = 2
Output: [1,2]
```
**Example 2**:
```shell
Input: nums = [1], k = 1
Output: [1]
```
**Example 3**:
```shell
Input: nums = [1,2,1,2,1,2,3,1,3,2], k = 2
Output: [1,2]
```
 
**Constraints**:
```shell
1 <= nums.length <= 10^5
-10^4 <= nums[i] <= 10^4
```
`k` is in the range `[1, the number of unique elements in the array]`.
It is guaranteed that the answer is **unique**.

### Reasoning


#### Step 1: Restate the problem:
I am given an integer array and integer `k`, and I should find the `k` frequent elements. Basicaly I need to count elements and return specified number most frequent elements.

- There is at least 1 element in the input array. The length is 1 to 500000.
- Negatives, zeros and positives exsit from -10000 to 10000, duplicates are also there
- Not sorted, output order does not matter

#### Step 2: Play examples and edge cases
- One element or all duplicates -> return this element in the output list
- The answer is unique, `k` is in range 1 to len(nums)

#### Step 3: Brute force

The brute force is what reveals _why_ the hash map matters.

Two sub-questions to separate out:

1. **Counting frequencies** — forget hash maps entirely for a second. If you could only use nested loops (no dict), how would you count how many times each number appears? Think back to how you counted characters in Valid Anagram's very first brute-force pass this morning — same idea, just on `nums` instead of a string. What would that nested-loop version look like, and what's its time complexity?
    
2. **Extracting the top k** — separately, suppose you _did_ already have a hash map of `{number: count}` (like your instinct suggests). "Just return the k most frequent ones" is hiding the actual work — how would you _naively_ pull the top k out of that map? What's the first, most obvious way, and what would it cost?

- 1. Yes I can use `.count()` method on list as well as on string type and count the number of a given element. So it would be one pass + count operation per element which is $O(n)$, total $O(n^2)$.
- 2. I can store those counts in the output list, sorted, and return just 'k' biggest. It will take $O(u)$ space to store counts of unique items and $O(u\space log\space u)$ time for sorting them by counts later, which can be ignored since total time anyway is quadratic $O(n^2)$ with nested `count()` method for each item.

**1. Sort + `count()` - Time $O(n^2)$, Space $O(n)$**

Sort is needed to group duplicates and skip them when counting.

[Visual](../images/347top_k_frequent_elem_brute1.png)

In [1]:
def brute1(nums: list[int], k: int) -> list[int]:
    # brute force - sort + one pass + count()
    # setup
    # sort input
    # set counts list: counts = []
    # scan sorted input

    # 1. skip duplicates
    # 2. build counts list of unique items
    # 3. sort counts items by their counts in desc order
    # 4. slice k items
    # 5. return k items
    counts = []
    nums.sort()

    for i in range(len(nums)):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        counts.append((nums[i], nums.count(nums[i])))
    return [t[0] for t in sorted(counts, key=lambda t: t[1], reverse=True)][:k]

Sorting first didn't remove the $O(n²)$ — you still call `.count()` once per unique value, each an $O(n)$ scan. Sorting only got you "duplicates are adjacent," but you didn't yet use that adjacency to _avoid_ `.count()`. That's exactly the piece your counter version needs to fix.

**2. Sort + counter flag - Time $O(n\space log\space n)$, Space $O(n)$**

[Visual](../images/347top_k_frequent_elem_brute2.png)

In [2]:
def brute2(nums: list[int], k: int) -> list[int]:
    # brute force - sort + one pass + counter flag (eliminate count() of O(n) time)
    # setup
    # sort input
    # set counter: counter = 1 (since at least 1 item exist)
    # set counts list: counts = []
    # scan sorted input - for loop - [1, len(nums)]

    # 1. if duplicate is hit (i = i - 1) -> increment counter: count
    # 2. otherwise distinct item is hit:
    #   a) write to counts list: [(i - 1, counter)]
    #   b) reset counter: counter = 1 (since another 1 distinct item found)
    # 3. last item handling: after loop write the last item: last item is either
    #   1 distinct item or the last of duplicates group
    # 3. sort counts items by their counts in desc order
    # 4. slice k items
    # 5. return k items

    nums.sort()
    counts = []
    counter = 1

    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1]:
            counter += 1
        else:
            counts.append((nums[i - 1], counter))
            counter = 1
    counts.append((nums[-1], counter))
    return [t[0] for t in sorted(counts, key=lambda t: t[1], reverse=True)][:k]

In [3]:
# brute2([2, 1, 3, 2, 1, 3, 1], 2)
brute2([1,2,1,2,1,2,3,1,3,2], 2)
brute2([1], 1)

[1]

In [1]:
d = {1: 3, 2: 2, 3: 1}

In [2]:
d.items()

dict_items([(1, 3), (2, 2), (3, 1)])

In [3]:
[k for k, _ in sorted(d.items(), key=lambda item: item[1], reverse=True)][:2]

[1, 2]

There are 2 branches in the loop: 1) duplicates count, where `counter` only grows with the duplicates group; 2) distinct item is found, the result of the **previous item group** is recorded and `counter` is reset. So, upon the loop end, we have `counter` either at the **last item of the duplicate group** or at the last **distinct item**. It safely is recorded after the loop with `nums[-1]` refering to the last item, along with `counter`, wich is either `number of duplicates` or `1` for the last distinct one.

`nums[-1]` is correct and it's the safer choice precisely because it doesn't depend on `i` being bound at all, so it **survives the single-element case** where the loop body never runs.

- **Time $O(n\space log\space n)$**: two sorts dominate the $O(n)$ linear pass and the $O(k)$ slice
- **Space $O(n)$**: worst case is all-distinct elements, `len(res) = n`, giving space $O(n)$. 

#### Step 4 - select algorithm

One-pass hash-map counting, no sorting `nums` at all.

**Compare it directly against what you just built. Two questions to settle before picking**:

1. The hash map counts in a single $O(n)$ pass, with no `log n` factor — where exactly does the `sort-first` approach pay a cost that the hash-map approach avoids?
2. You'll still need to get the top `k` out of the counts afterward. Is a full sort of all the counts (`O(u log u)`) actually necessary, given that a frequency can only ever range from `1` to `n` — or is there a way to exploit that bounded range to skip a general sort?

- 1. I don't have to sort nums since I use hash map to store an element and its count along the way, I either add a new item to the hash map or increment its existing count, so I don't nee the input nums be sorted, duplicates fall into inrementing the count. Ths sort-first approach pays log n factor giving duplicates adjacent for the counter to make a job.
- 2. For now I only think about sorting the `u` counted items in the hash map, with something like this:
`[k for k, _ in sorted(d.items(), key=lambda item: item[1], reverse=True)][:k]`, it is still the same approach as in the previous one. At the moment I don't know another approach to avoid sorting to fetch `k` most frequent items out of the hash map.

**Good on point 1 — hash-map counting genuinely eliminates the need to sort `nums`**.

But look closer at point 2, because I don't think it gets you as far as it seems. Your hash-map counting is $O(n)$, sure — but then you still do `sorted(d.items(), key=..., reverse=True)`, which is $O(u\space log\space u)$.

Worst case: what is `u` when every element in `nums` is distinct? And in that case, what does $O(u\space log\space u)$ become? Now compare that to the $O(n\space log\space n)$ total of your _original_ sort-`nums` approach — are these two approaches actually different in Big-O terms, or do they land on the exact same worst-case complexity?

- `u = n` if all items are distinct, in this case it is $O(n\space log\space n)$, and yes this approach is the same time complexity as the sort-first above. I need to find a way to avaoid sorting. I said that frequency can only ever range from `1` to `n`, that is true. The task is to return `k` most frequent elements. So, counts are values in the hash map, I need to find `k` keys with the biggest values. The hash map is filled in with values and counts from unordered list. For now I'm here in my reasoning timeline.

**That's the right place to be stuck — this is where the insight actually lives**. Here's a nudge toward it, since you've hit the wall on how to avoid a comparison sort entirely.

You already know something useful:
> frequency (the value in your hash map) is bounded — it can only be an integer **from `0` to `n`**.

Comparison-based sorting (`sorted()`) is the general-purpose tool you reach for when keys could be _anything_. But when keys are bounded integers, there's a different family of technique that sidesteps comparisons altogether — you may have run into "counting sort" or "bucket sort" by name before.

> Think about it this way: instead of asking "how do I sort these `(value, count)` pairs by count," what if you flipped the relationship — used the _count_ itself as an index into something, rather than as something you compare?

Concretely: imagine an array where the position `i` holds _all the values that appeared exactly `i` times_.

**How many positions would that array need, given counts range `0` to `n`**? And if you walked through your hash map once, placing each value into its corresponding position — what would that cost, in terms of Big-O?

Sketch out what that structure would look like on `nums = [1,1,1,2,2,3], k=2` — what's in each position?
- So, yes indeed I need to use counts as indexes (`from `0` to n`), counts can never be more than `n` and no negatives. Counts will point to not a single value but to a **bunch of values that happened to occure a number of times**. There must be a data structure, or a container to store a collection which occured serveral times. Counts are shared. So it could be a list for instance under a specific index.

Exactly — a list under each index, since multiple distinct values can share the same frequency.
> That's the bucket-sort structure: `buckets[count]` holds every value that occurred `count` times.

That completes Step 4's selection. Before we move to writing any code (Step 5 — function signature), state it out loud:

- **The key insight in one sentence** — why does bucketing by count let you skip comparison-based sorting entirely?
- **Target time complexity** — building the hash map, building the buckets, and then walking the buckets from highest frequency down to collect `k` values — what's the total, and why?
- **Target space complexity** — for the hash map plus the bucket array plus the result?

Answer:
- The key insight is: bucketing by count let's me to skip comparison-based sorting since I sort elements by a bucket using a list container, I just put an element to its corresponding bucket indexed by counts, there is no comparison element at all.
- After bucket sorting, I could go from the highest index bucket (highest frequency) and slice `k` elements to return.
- The time complexity is:
  - building the hash map is $O(n)$
  - building the buckets is constant $O(1)$ - I append a new count (index) with its inital value to the mapping internal list, then append to this list over an over as needed.
- The total time is dominating $O(n)$. For space, I first build a hash map of unique keys (elements as keys mapping to their counts) - $O(u)$, then building buckets to store again, elements from `1` to `n`, so it's $O(n)$ dominating over $O(u)$, and the result is still $O(k)$ that could of course be anything up to $O(n)$.

Good insight sentence, and close on complexity, but two things need sharpening.

**Building the buckets** isn't $O(1)$ overall — each individual append is $O(1)$, but you're doing that once per unique entry in the hash map, so the _total_ cost of filling the buckets is $O(u)$, not $O(1)$. And there's a cost you haven't counted yet: creating the buckets array itself — something like `buckets = [[] for _ in range(n + 1)]` — has to allocate `n+1` empty lists. What's the cost of _that_ allocation alone, before you even put anything in it?

**The part you haven't addressed yet**: walking from the highest-index bucket down to collect `k` values. Consider this case — `nums` has `n` elements, all distinct (so every count is `1`). That means `buckets[1]` holds all `n` values, and `buckets[2]` through `buckets[n]` are all empty. To reach `buckets[1]`, how many bucket positions does your walk have to pass through, even though most of them are empty? What does that make the worst-case cost of the walk-and-collect phase?

Once you've reasoned through both of those, does the overall time complexity change from what you stated, or does it land in the same place for a different reason?

#### Step 5, 6 - signature and pseudocode

[Visual](../images/347top_k_frequent_elem_hashmap.png)

```py
def top_k_freq_elem(nums: list[int], k: int) -> int:
    # hash map + bucket sort
    # setup
    # set counts hash map: counts = {}
    # set bucket list: buckets = [[] for _ in range(n+1)]
    # scan input - for loop - full pass

    # 1. build counts
    # 2. build buckets - for loop over counts
    # 3. build result - scan buckets from highest index and slice 'k' items
    pass
```

#### Step 7 - write code

In [ ]:
def top_k_freq_elem(nums: list[int], k: int) -> list[int]:
    counts = {}
    buckets = [[] for _ in range(len(nums) + 1)]  # n + 1; space O(n) / time O(n)
    res = []

    # 1. build counts - time O(n) / space O(n)
    for i in range(len(nums)):
        counts[nums[i]] = counts.get(nums[i], 0) + 1

    # 2. build buckets - time O(u) / space O(n)
    for key, value in counts.items():
        buckets[value].append(key)

    # 3. build result - time O(n) / O(k)
    for i in range(len(buckets) - 1, -1, -1):
        bucket = buckets[i]
        if bucket:
            for item in bucket:
                if len(res) >= k:
                    return res
                res.append(item)
        # if buckets[i]:
        #     for j in range(len(buckets[i])):
        #         if len(res) >= k:
        #             return res
        #         res.append(buckets[i][j])
    return res

In [9]:
# top_k_freq_elem([1, 1, 1, 2, 2, 3], 2)
top_k_freq_elem([1, 1, 2, 2, 3], 1)
# nums = [1, 2, 3, 4, 5, 6]
# buckets = [[], [1, 2, 3, 4, 5, 6], [], [], [], []]

[1]

`n+1` question: it's not really about index `0` being wasted (though that's true too) — it's about the _top_ end. A count can be as large as `n` itself (e.g. every element in `nums` is the same value). A list of size `n` only has valid indices `0` through `n-1` — it has no slot for index `n`. So you need `n+1` slots specifically to make room for the _maximum possible count_, not just to skip index `0`.

**Time & Space Complexity**

State the final complexity for this corrected version — does the early return `res` inside the inner loop change the $O(n)$ time / $O(n)$ space you derived at Step 4, or does the worst case stay the same?
- **Time**: $O(n)$
- **Space**: $O(n)$

1. Building initial buckets list of length `n+1` takes $O(n)$ time and space
2. Building counts hash map of unique keys takes $O(n)$ time and space
3. Building actual buckets list: the `n+1` empty list slots **were already accounted** for in step 1's allocation. Loop (`for key, value in counts.items()`) only appends — and it runs once per unique key, so it adds `u` items total, not `n`. The reason total space still comes out to $O(n)$ isn't because step 3 stores n items — it's because step 1's skeleton array itself already costs $O(n)$, regardless of how many items eventually get appended into it.
4. Building result list takes up to $O(n)$ time in the worst case when all items are in the first bucket, and $O(k)$ space since we store only required k items in the resulting list.

> This is bucket sort / counting-sort-by-frequency pattern — file it as the go-to whenever you need "top/bottom k by frequency (or any bounded-integer key)" without paying a comparison-sort's log factor.

Try Top K Frequent Words (LeetCode 692) alone before next session — same bucket idea, but the tie-breaking rule (alphabetical order within a frequency) will force you to reconsider whether a plain list-per-bucket is still enough.

With that, this problem's closed: **brute force** $O(n²)$ → **sort-based** $O(n\space log\space n)$ → **hash map + bucket sort** $O(n)$, each step driven by removing a specific piece of wasted work (recomputed counts, then the comparison-sort's `log` factor). Bucket sort by frequency — file it, and Top K Frequent Words is still the one to try solo before next session.

---

In [ ]:
def top_k_frequent_elements(nums: list[int], k: int) -> list[int]:
    counts = {}

    for num in nums:
        # increment existing count or set 1 for the very first adding
        counts[num] = 1 + counts.get(num, 0)  # {int: count}

    # store count result in array and sort in asc
    array = []
    for num, cnt in counts.items():
        array.append([cnt, num])  # [[count, int], [count, int]]

    array.sort()

    # build resulting array
    result = []
    while len(result) < k:
        # take int value from the end of array and store in result
        result.append(array.pop()[1])

    return result


print(top_k_frequent_elements([1,2,1,2,1,2,3,1,3,2], 2))

In [ ]:
# neetcode
def top_k_frequent_elements(nums: list[int], k: int):
    from collections import defaultdict
    counts = defaultdict(int)

    for i in range(len(nums)):
        counts[nums[i]] += 1

    print(counts)
    sorted_counts = sorted(counts.items(), reverse=True, key=lambda item: item[1])
    return [item[0] for item in sorted_counts][:k]


print(top_k_frequent_elements([1, 2, 2, 2, 1, 4], 2))